# Deep-layer lithography: into the resist volume

The quickstart stopped at the 2D aerial image. Thick resists and non-projection
processes need the dose **through the depth of the film**. This notebook covers:

1. **LIGA** deep X-ray lithography: depth dose, absolute exposure time and beam
   filtering for a bending-magnet source;
2. **LIGA edge diffraction**: the Fresnel straight edge vs the legacy Gaussian
   blur, and the sidewall through 500 µm of PMMA;
3. a **laboratory X-ray tube** as a LIGA source (absolute flux, hours of exposure);
4. **multi-beam interference**: a standing-wave lattice recorded in one shot;
5. **volumetric exposure + bake + level-set development** of a projected image.

Model status (from the [capability matrix](../../docs/capability-matrix.md)):
LIGA depth dose and exposure time ✅, LIGA proximity diffraction ✅/🔶,
interference ✅/🔶, volumetric exposure 🔶, level-set development ✅/🔶.
Talbot and EUV interference lithography are in notebook 06.

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import viz

%matplotlib inline

## 1. LIGA: depth dose and absolute exposure time

LIGA shadow-prints a gold absorber mask into hundreds of microns of PMMA with
hard X-rays from a synchrotron bending magnet. The dose falls with depth because
PMMA absorbs the softer part of the spectrum first. Two dose limits frame the
process: the **bottom** of the resist must receive enough dose to develop
(here 3 kJ/cm³), while the **top** must stay below the damage ceiling
(about 20 kJ/cm³, where PMMA foams and cracks).

`simulate_liga` integrates the bending-magnet spectrum (photons per log-energy
bin) through the mask membrane, optional beam filters and the resist, using NIST
attenuation $\mu$ for transport and the energy-absorption coefficient $\mu_{en}$
for deposition. With the ring parameters and the scan geometry it also returns
the **absolute exposure time**. A filter hardens the beam: it absorbs the soft
photons that would overdose the top, at the cost of a longer exposure.

The Gaussian edge model and a coarse grid keep this cell fast; the depth dose and
the exposure time do not depend on the lateral model.

In [ ]:
bm = huv.SourceConfig.synchrotron_liga_bending_magnet()      # 2.5 GeV, 1.5 T, 200 mA
fig, ax = plt.subplots(figsize=(8, 4.5))
for filters, label in ((None, "no filter"), ([("Be", 100.0)], "100 µm Be"), ([("Al", 20.0)], "20 µm Al")):
    liga = huv.simulate_liga(
        source=bm, source_distance_m=15.0, vertical_scan_mm=50.0, filters=filters,
        resist_thickness_um=500.0, grid_size=32, pixel_nm=200.0, nz=16,
        diffraction="gaussian", warn=False,
    )
    z_um, dose = liga.depth_dose
    ax.plot(z_um, dose, label=f"{label}: top {liga.top_dose_kj_cm3:.1f} kJ/cm³, {liga.exposure_time_s:.0f} s")
    print(f"{label:10s} top/bottom {liga.dose_ratio:5.2f}  exposure {liga.exposure_time_s:5.0f} s  "
          f"({liga.exposure_charge_ma_h:.1f} mA·h)  above damage ceiling: {liga.exceeds_damage_ceiling}")
ax.axhline(20.0, ls="--", color="grey", lw=1)
ax.text(5, 21, "damage ceiling 20 kJ/cm³", color="grey", fontsize=9)
ax.set(yscale="log", xlabel="depth in PMMA (µm)", ylabel="absorbed dose (kJ/cm³)",
       title="LIGA depth dose, 500 µm PMMA, bottom dose 3 kJ/cm³")
ax.legend()
ax.grid(alpha=0.3, which="both")
print("model status:", liga.status)

## 2. The edge: Fresnel diffraction, not a Gaussian blur

Between the mask and the resist top there is a proximity gap $g$ (here 100 µm).
Diffraction at the absorber edge then has the Fresnel scale $\sqrt{\lambda g}$,
a few hundred nanometres at 8 keV. The engine propagates the complex field
behind the absorber with the angular spectrum (the default `diffraction="fresnel"`),
which reproduces the textbook knife-edge pattern: 1/4 of the open-field
intensity at the geometric edge, ringing on the open side with a first maximum
about 37 % above the open level. The legacy model replaced this by a Gaussian
blur of width $\frac{1}{2} \sqrt{\lambda g}$, which has no ringing and a wider transition.

`liga_edge_profile` evaluates the analytic Fresnel edge directly, at any
resolution. Below: a monochromatic 8 keV edge, then the polychromatic
bending-magnet edge at several depths, which gives the sidewall angle.

In [ ]:
mono = [(8.0, 1.0)]
edge = huv.liga_edge_profile(
    spectrum_table=mono, absorber_thickness_um=200.0, membrane_thickness_um=0.0,
    proximity_gap_um=100.0, depths_um=[0.0], dx_nm=1.0, x_min_nm=-400.0, x_max_nm=400.0,
)
x = np.asarray(edge.x_nm)
rel = np.asarray(edge.dose_kj_cm3)[0] / float(np.asarray(edge.open_dose_kj_cm3)[0])
lam_nm = 1.23984193 / 8.0
sigma = 0.5 * np.sqrt(lam_nm * 100e3)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.2))
ax0.plot(x, rel, label="Fresnel (default)")
gauss = 0.5 * (1 + np.vectorize(math.erf)(x / (np.sqrt(2) * sigma)))   # Gaussian-blurred step
ax0.plot(x, gauss, label="legacy Gaussian blur")
ax0.axvspan(x.min(), 0, color="grey", alpha=0.15)
ax0.text(-390, 1.3, "Au absorber", fontsize=9)
ax0.set(xlabel="x from the absorber edge (nm)", ylabel="dose / open-field dose",
        title=f"8 keV, 100 µm gap: sqrt(lambda g) = {np.sqrt(lam_nm * 100e3):.0f} nm")
ax0.legend(loc="center right")
ax0.grid(alpha=0.3)
print(f"value at the geometric edge: {np.interp(0.0, x, rel):.3f} (Fresnel theory 0.25)")
print(f"first maximum: {rel.max():.3f} (Fresnel theory 1.37)")

profile = huv.liga_edge_profile(source=bm, depths_um=[0.0, 125.0, 250.0, 375.0, 500.0],
                                dx_nm=2.0, x_min_nm=-600.0, x_max_nm=600.0)
viz.plot_liga_edge_profile(profile, ax=ax1)
fig.tight_layout()
print(f"edge positions at 2.5 kJ/cm³: {np.round(profile.edge_positions_nm(2.5), 1)} nm")
print(f"sidewall angle from vertical: {profile.sidewall_angle_deg(2.5):.4f} deg   ({profile.status})")

The sidewall is vertical to within hundredths of a degree over 500 µm: this is
why LIGA reaches aspect ratios of 100 and more. The model is a thin-screen
absorber without photoelectron or fluorescence transport (an optional Gaussian
photoelectron blur exists), hence the 🔶 on the lateral part.

## 3. A laboratory X-ray tube as a LIGA source

Without a synchrotron, a rotating-anode or sealed tube can expose thinner PMMA.
The tube model (lines, edges, Duane-Hunt cutoff and Be window ✅; Kramers
continuum and absolute flux 🔶) provides an **absolute** spectral flux density at a
given distance, which `simulate_liga(flux_density=...)` turns into an exposure
time. A 60 kV / 30 mA tungsten tube at 100 mm needs on the order of two days for
200 µm of PMMA, compared with minutes at a synchrotron.

In [ ]:
tube = huv.SourceConfig.xray_tube(anode="W")
flux = tube.spectral_flux_density(distance_mm=100.0, n_bins=200)
liga_tube = huv.simulate_liga(
    flux_density=flux, resist_thickness_um=200.0, cd_nm=20000.0, pitch_nm=40000.0,
    grid_size=32, pixel_nm=2500.0, nz=16, diffraction="gaussian", warn=False,
)
print(f"W tube, 100 mm, 200 µm PMMA: exposure {liga_tube.exposure_time_s / 3600:.0f} h, "
      f"top/bottom {liga_tube.dose_ratio:.2f}, mean photon energy top {liga_tube.mean_energy_top_kev:.1f} keV "
      f"({liga_tube.status})")

## 4. Multi-beam interference: a standing-wave lattice

Interference (holographic) lithography superposes a few coherent plane waves
into a periodic intensity pattern and records it as a photoactive-compound (PAC)
volume in one exposure. For two beams at half-angle θ in a medium of index n the
fringe period is $\lambda / (2 n \sin \theta)$. We slice the PAC volume in the
x-z plane: m = 1 unexposed, m → 0 fully exposed.

In [ ]:
vol = huv.simulate_interference(
    preset="two_beam", wavelength_nm=200.0, n_medium=1.6, half_angle_deg=30.0,
    nx=96, ny=8, nz=64, x_span_nm=1200.0, y_span_nm=100.0, z_span_nm=600.0, dose_scale=12.0,
)
ax = viz.plot_xz_slice(vol, title="Two-beam interference: PAC in the x-z plane", colorbar_label="PAC m")
pac = np.asarray(vol.values)
print(f"expected fringe period {200.0 / (2 * 1.6 * np.sin(np.radians(30.0))):.1f} nm; "
      f"PAC range {pac.min():.3f} .. {pac.max():.3f}")

## 5. Volumetric exposure, bake and level-set development

For a projected image the dose varies in x, y **and** z: the aerial image
defocuses through the resist and the thin-film stack sets up a standing wave
between the air interface and the substrate. `simulate_volumetric` chains:

- `expose_volumetric` (🔶): the aerial image at several depth planes ×
  the exact thin-film intensity profile, with split-step Dill bleaching;
- a post-exposure bake: here an anisotropic Gaussian (✅, 20 nm lateral, 10 nm
  vertical), which smooths the standing-wave ripple;
- development: here the **level set** (✅/🔶), which moves the resist front with
  the dissolution rate re-evaluated at the front, including surface inhibition
  (a slower top skin).

We image 150 nm lines on a 300 nm pitch (F$_2$, NA 0.75) into the default 150 nm
resist on silicon at 24 mJ/cm² and develop for 20 s. The dose deposited in the film
scales with the resist index ($n_r \lvert E \rvert^2$ relative to the incident medium), so a
resist with n = 1.65 absorbs 1.65 times the dose a unit-index model would predict.

In [ ]:
source = huv.SourceConfig.f2_laser(sigma=0.7)
optics = huv.OpticsConfig(numerical_aperture=0.75)
mask = huv.MaskConfig.line_space(150.0, 300.0)
grid = huv.GridConfig(size=64, pixel_nm=600.0 / 64)              # 2 pitches
proc = huv.simulate_volumetric(
    source, optics, mask, huv.FilmStackConfig(), huv.ResistConfig(), grid,
    dose_mj_cm2=24.0, nz=32, n_defocus_planes=4, dose_steps=5,
    peb="gaussian", peb_lateral_nm=20.0, peb_vertical_nm=10.0,
    develop="level_set", dev_time_s=20.0, surface_rate_ratio=0.3, inhibition_depth_nm=15.0,
)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
viz.plot_xz_slice(proc.latent, ax=axes[0], title="Latent PAC (after exposure)", colorbar_label="m")
viz.plot_xz_slice(proc.baked, ax=axes[1], title="After the anisotropic bake", colorbar_label="m")
viz.plot_development_front({"level set": proc.level_set.arrival_times}, 20.0, ax=axes[2])
fig.tight_layout()
print("developed CD (resist line) at the top, middle and bottom slice:",
      [round(float(c), 1) for c in np.asarray(proc.developed_cd_nm)[[0, len(proc.developed_cd_nm) // 2, -1]]], "nm")
print("level-set status:", proc.level_set.status)

The latent image shows the standing-wave layers (the period is $\lambda / (2 n)$ in
the resist). A 10 nm vertical diffusion length only softens them, so the
developed sidewall keeps its scallops; a longer vertical bake (try
`peb_vertical_nm=30.0`) smooths them further. The development front clears the
spaces to the substrate while the lines stand. The "CD" here is the width of the
remaining resist line in each depth slice, quantized to the 9.4 nm pixel.

### API vs. CLI

The same deep-layer processes run from TOML configs:

```bash
highuvlith deep --config examples/liga.toml
highuvlith deep --config examples/volumetric.toml --output profile.png
highuvlith deep --config examples/talbot.toml
```

Docs: [LIGA / deep X-ray](../../docs/processes/liga-deep-xray.md),
[Volumetric exposure](../../docs/processes/volumetric-exposure.md),
[Resist models](../../docs/processes/resist-models.md),
[Interference](../../docs/processes/interference-volumetric.md).